# Exposure-response — the Phase B machinery

**NOT FOR CLINICAL USE.** Population/trial-level forward simulation only. Executed in CI (nbmake).

An exposure-response (ER) record maps a PK exposure metric `C` (e.g. average steady-state concentration) to the drug-effect magnitude `E` that drives a TGI model's kill term. This completes the chain **PK → exposure → tumor dynamics → survival**, and is the seam where a Hypnos PK record composes with an Onkos TGI model.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import onkos
from onkos.export.reference import effect
from onkos.export.registry import get_kernel, kernel_values

ds = onkos.load()
er_ids = [r.id for r in ds if r.purpose == 'exposure_response']
print('exposure-response records:', er_ids)

In [ ]:
# Emax is half-maximal exactly at EC50; effect is bounded by Emax.
r = ds['exposure_response.emax_generic']
v = kernel_values(r)
assert abs(float(effect(get_kernel(r), v['EC50'], v)) - v['Emax']/2) < 1e-9

C = np.linspace(0, 600, 200)
for rid in er_ids:
    rr = ds[rid]
    plt.plot(C, effect(get_kernel(rr), C, kernel_values(rr)), label=rid.split('.', 1)[1])
plt.xlabel('exposure C (ug/L)'); plt.ylabel('effect E'); plt.legend(fontsize=7); plt.title('C -> E');

In [ ]:
# Constant exposure drives the Claret TGI; higher exposure -> deeper response, longer OS.
ctx = {'tumor_type': 'NSCLC', 'line': 'first'}
er = 'exposure_response.dacomitinib_egfr.emax'
low = onkos.simulate(ds, 'resistance.claret_2009.tgi', context=ctx, exposure=50.0, exposure_response=er)
high = onkos.simulate(ds, 'resistance.claret_2009.tgi', context=ctx, exposure=400.0, exposure_response=er)
print('low  exposure: depth', round(low.metrics['depth_of_response'], 3), 'median OS', round(low.median_os, 1))
print('high exposure: depth', round(high.metrics['depth_of_response'], 3), 'median OS', round(high.median_os, 1))
assert high.median_os > low.median_os

In [ ]:
# A time-varying PK profile (Hypnos-style) integrates the tumor ODE with E(t).
t = np.linspace(0, 104, 209)
decaying = 300.0 * np.exp(-0.02 * t)
pk = onkos.simulate(ds, 'resistance.claret_2009.tgi', context=ctx,
                    exposure=decaying, exposure_response='exposure_response.emax_generic', t=t)
print('PK-driven tier:', pk.tier, '| week8:', round(pk.metrics['week8_relative_change'], 3))
assert pk.tumor_size.shape == t.shape and np.all(pk.tumor_size > 0)

# Out-of-context ER floors the result to D with a warning.
oc = onkos.simulate(ds, 'resistance.claret_2009.tgi', context={'tumor_type': 'breast', 'line': 'first'},
                    exposure=200.0, exposure_response=er)
print('breast-context tier:', oc.tier)
assert oc.tier == 'D'